In [ ]:
# 1/4 - clone main + prototype branch (co kernel .tkv -> C)
import subprocess, os, shutil
print('===== NATIVE LOSS AFTER TKV C REPLACEMENT =====')
for d,br in [('/content/TVmain','main'),('/content/TVcpu','prototype/cuda-backend')]:
    subprocess.run('rm -rf '+d, shell=True)
    subprocess.run('git clone --depth 1 --branch '+br+' https://github.com/nguyenhungtran18/TokenVector.git '+d, shell=True, capture_output=True, text=True, timeout=600)
    print(br, subprocess.check_output('git -C '+d+' rev-parse --short HEAD', shell=True).decode().strip())
subprocess.run('rm -rf /content/TVinf', shell=True)
subprocess.run('git clone --depth 1 https://github.com/nguyenhungtran18/TokenVector.Inference.git /content/TVinf', shell=True, capture_output=True, text=True, timeout=600)

In [ ]:
# 2/4 - replace tkv_linear_dw.c by .tkv-generated C, build native .so
import subprocess, shutil, os
mainroot='/content/TVmain/Release/tools/simd_native'
gen='/content/TVcpu/Release/cuda_backend/examples/linear_dw_cpu.cpu.c'
shutil.copyfile(gen, mainroot+'/tkv_linear_dw.c')
print('replaced C source, bytes=',os.path.getsize(mainroot+'/tkv_linear_dw.c'))
srcs='tkv_simd_native.c tkv_cuda_native.c tkv_linear_dw.c tkv_bwd_ops.c tkv_flash_attn.c tkv_step124.c tkv_prof.c tkv_attn_cuda.c'
p=subprocess.run('cd '+mainroot+' && gcc -O2 -mavx2 -mfma -shared -fPIC '+srcs+' -lm -lpthread -ldl -o tkv_simd_native.so', shell=True, capture_output=True, text=True, timeout=600)
print('native build rc=',p.returncode)
if p.returncode!=0: print(p.stderr[:3000])
# copy .so to Inference demo (where bseq.dll loads it)
shutil.copyfile(mainroot+'/tkv_simd_native.so','/content/TVinf/demo/tkv_simd_native.so')
print('so copied', os.path.exists('/content/TVinf/demo/tkv_simd_native.so'))

In [ ]:
# 3/4 - run native fullstep (mode 1) and read loss
import subprocess, os, re
cfg='/content/TVinf/tv/tests/_seq_cfg.txt'
open(cfg,'w').write('1024\n1\n1\n')
env=dict(os.environ); env.pop('TKV_TORCH_GPU',None)
p=subprocess.run('cd /content/TVinf && LD_LIBRARY_PATH=/content/TVinf/demo:/usr/local/lib/python3.13/dist-packages/torch/lib /opt/dotnet/dotnet demo/bseq.dll',
                 shell=True, capture_output=True, text=True, timeout=1800, env=env)
print('bseq rc=',p.returncode)
print('STDOUT tail:',(p.stdout or '')[-1500:])
print('STDERR head:',(p.stderr or '')[:600])

In [ ]:
# 4/4 - summary
print('EXPECTED native loss = 25.454 (EXACT). If differs -> .tkv C not usable for native path.')